# Generating Dataset for FB_INF project

The current notebook is a simple way of randomly selecting images from the ImageNet dataset for the desired classes in the FB_INF experiment for Paulo's first research project.

The code presented here is super simple. I have it on a separate notebook for ease of use and organization

In [1]:
#Loading the directories needed

%load_ext autoreload
%autoreload 2

import os
import shutil
import numpy as np

from wormholes import *
from wormholes.perturb import *
from wormholes.perturb.gen_v3 import GenV3

In [2]:
#When working with the jupyter network and the cluster, it sometimes fails to find the right directories. 
# I thus ensure that they are properly set

# Set the HOME environment variable to your current working directory
os.environ["HOME"] = "/project/3018078.01/Gaziv/Wormholes_FB"

# Now the `os.path.expanduser("~")` will resolve to this directory
print(os.path.expanduser("~")) 

/project/3018078.01/Gaziv/Wormholes_FB


In [3]:

#Selecting the dataset I want to use for trasnferring the images from ilrsvc to my own folders
class_dict = CLASS_DICT['RestrictedImageNet'] #maps class labels to human-readble names for Restricted ImageNet

#because the dictionary containing the info to pass the RIN and imagenet images to my folders is a function of the class Genv3
#I am creating an instance of the class to get to use the features
instance = GenV3()

# Access the data dictionary
data_dict = instance.data_dict

# Print or inspect data_dict (not needed, but useful)
print(data_dict.keys())  # To see available classes
print(len(data_dict))  # To see how many classes there are


dict_keys(['frog', 'turtle', 'lizard', 'bird', 'crab', 'dog', 'cat', 'bear', 'insect', 'rabbit', 'gazelle', 'primate'])
12


In [17]:
#define the directory for the output (i.e., where the images will get saved)
output_folder_root = "/project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo"

#make a function to sample images. (not fully my function). 
#for the specific n_sample_class, you will always get the same images if ask for the same number. 
#I ensure that every (class, sample) pair gets a unique seed.
# i (class index), n_sample_class (the number of images to select per class), j (the image index within the class (ranges from 0 to n_sample_class - 1)).

def sample_images(output_dir=output_folder_root, n_sample_class=1):
    """Randomly samples n_sample_class images per class and saves them to output_dir."""
    os.makedirs(output_dir, exist_ok=True)

    for i, (class_name, image_paths) in enumerate(data_dict.items()):
        class_output_dir = os.path.join(output_dir, class_name)
        os.makedirs(class_output_dir, exist_ok=True)

        rng = np.random.RandomState(i)

        # Sample without replacement to ensure diverse selection
        sampled_paths = rng.choice(image_paths, size=min(n_sample_class, len(image_paths)), replace=False)

        for img_path in sampled_paths:
            shutil.copy(img_path, os.path.join(class_output_dir, os.path.basename(img_path)))
            print(f"Copied {img_path} -> {class_output_dir}")

# Example Usage
sample_images(n_sample_class=66)


Copied /project/3018078.01/Gaziv/Wormholes_FB/data/ilsvrc/val/n01644900/ILSVRC2012_val_00013354.JPEG -> /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog
Copied /project/3018078.01/Gaziv/Wormholes_FB/data/ilsvrc/val/n01644373/ILSVRC2012_val_00015245.JPEG -> /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog
Copied /project/3018078.01/Gaziv/Wormholes_FB/data/ilsvrc/val/n01641577/ILSVRC2012_val_00028352.JPEG -> /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog
Copied /project/3018078.01/Gaziv/Wormholes_FB/data/ilsvrc/val/n01644900/ILSVRC2012_val_00006830.JPEG -> /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog
Copied /project/3018078.01/Gaziv/Wormholes_FB/data/ilsvrc/val/n01641577/ILSVRC2012_val_00010287.JPEG -> /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog
Copied /project/3018078.01/Gaziv/Wormholes_FB/data/ilsvrc/val/n01644900/ILSVRC2012_val_00000037.JPEG -> /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo/frog
Copied /project/301807

In [18]:

# Define the directory for the output
output_folder_root = "/project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo_test"

def sample_images(output_dir=output_folder_root, n_sample_class=1):
    """Randomly samples n_sample_class images per class and saves them to output_dir."""
    os.makedirs(output_dir, exist_ok=True)

    for i, (class_name, image_paths) in enumerate(data_dict.items()):
        class_output_dir = os.path.join(output_dir, class_name)
        os.makedirs(class_output_dir, exist_ok=True)

        if not image_paths:
            print(f"Skipping {class_name}: No images available.")
            continue

        # Initialize random state per class
        rng = np.random.RandomState(i)

        # Shuffle images before sampling
        shuffled_paths = image_paths.copy()
        rng.shuffle(shuffled_paths)

        # Sample without replacement (handling cases where n_sample_class > available images)
        sampled_paths = shuffled_paths[:min(n_sample_class, len(shuffled_paths))]

        for img_path in sampled_paths:
            if not os.path.exists(img_path):
                print(f"Warning: {img_path} not found, skipping.")
                continue
            try:
                shutil.copy(img_path, os.path.join(class_output_dir, os.path.basename(img_path)))
                print(f"Copied {img_path} -> {class_output_dir}")
            except Exception as e:
                print(f"Error copying {img_path}: {e}")

# Example Usage
sample_images(n_sample_class=50)


Copied /project/3018078.01/Gaziv/Wormholes_FB/data/ilsvrc/val/n01644900/ILSVRC2012_val_00013354.JPEG -> /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo_test/frog
Copied /project/3018078.01/Gaziv/Wormholes_FB/data/ilsvrc/val/n01644373/ILSVRC2012_val_00015245.JPEG -> /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo_test/frog
Copied /project/3018078.01/Gaziv/Wormholes_FB/data/ilsvrc/val/n01641577/ILSVRC2012_val_00028352.JPEG -> /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo_test/frog
Copied /project/3018078.01/Gaziv/Wormholes_FB/data/ilsvrc/val/n01644900/ILSVRC2012_val_00006830.JPEG -> /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo_test/frog
Copied /project/3018078.01/Gaziv/Wormholes_FB/data/ilsvrc/val/n01641577/ILSVRC2012_val_00010287.JPEG -> /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo_test/frog
Copied /project/3018078.01/Gaziv/Wormholes_FB/data/ilsvrc/val/n01644900/ILSVRC2012_val_00000037.JPEG -> /project/3018078.01/Gaziv/Wormholes_FB/data/OOD_Paulo_te